In [ ]:
!nvidia-smi

Wed Sep  9 11:21:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -la /content/drive/MyDrive/silent_speech
!echo "--- hybrid fine-tune runs ---" && ls -la /content/drive/MyDrive/silent_speech/output_finetune_hybrid

total 10612161
-rw------- 1 root root 6866471136 Aug 24 03:45 emg_dataset.h5
-rw------- 1 root root 3919507637 Aug 24 03:45 emg_data.tar.gz
-rw------- 1 root root   18261251 Sep  8 12:18 hybrid_warmstart.pt
drwx------ 2 root root       4096 Aug 24 03:49 KenLM
drwx------ 2 root root       4096 Aug 24 03:50 output
drwx------ 2 root root       4096 Aug 31 05:44 output_finetune
drwx------ 2 root root       4096 Sep  8 12:19 output_finetune_hybrid
drwx------ 2 root root       4096 Sep  8 12:07 output_finetune_stage1
-rw------- 1 root root   18268628 Sep  8 11:40 resume_finetune.pt
-rw------- 1 root root   18268278 Sep  9 10:14 resume_hybrid.pt
-rw------- 1 root root   10513993 Aug 31 04:28 resume.pt
-rw------- 1 root root   15539046 Sep  8 12:07 TinyMyo_backbone.pt
--- hybrid fine-tune runs ---
total 107076
-rw------- 1 root root 18273881 Sep  8 12:20 model_20260908_121901_best.pt
-rw------- 1 root root 18273881 Sep  8 12:25 model_20260908_121901_last.pt
-rw------- 1 root root 18273881 Sep 

In [ ]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech
!git submodule update --init text_alignments
!tar -xzf text_alignments/text_alignments.tar.gz
!sed -i '/norm_layer=norm_layer,/d' architecture.py

/content
Cloning into 'silent_speech'...
remote: Enumerating objects: 531, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (224/224), done.
remote: Total 531 (delta 216), reused 270 (delta 138), pack-reused 156 (from 1)
Receiving objects: 100% (531/531), 7.30 MiB | 19.42 MiB/s, done.
Resolving deltas: 100% (285/285), done.
/content/silent_speech
Submodule 'text_alignments' (https://github.com/dgaddy/silent_speech_alignments.git) registered for path 'text_alignments'
Cloning into '/content/silent_speech/text_alignments'...
Submodule path 'text_alignments': checked out '5c71ae9fcbb94e74e19eb9547c3b404baf6126a7'


In [ ]:
!pip install -q flashlight-text jiwer timm torchinfo torchprofile wandb tensorboard \
  librosa soundfile noisereduce resampy praat-textgrids unidecode \
  h5py scipy joblib matplotlib tqdm requests numpy huggingface_hub safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

In [ ]:
%env DATA_PATH=/content/data

env: DATA_PATH=/content/data


In [ ]:
%cd /content/silent_speech
!mkdir -p /content/data/Gaddy/h5
!cp /content/drive/MyDrive/silent_speech/emg_dataset.h5 /content/data/Gaddy/h5/ && echo "h5 copied" || echo "!! h5 NOT on Drive"
!cp -r /content/drive/MyDrive/silent_speech/KenLM /content/silent_speech/ && echo "KenLM copied" || echo "!! KenLM NOT on Drive"
!cp /content/drive/MyDrive/silent_speech/emg_data.tar.gz /content/data/Gaddy/ 2>/dev/null && echo "tar restored" || echo "no tar on Drive; downloading fresh"
!python download_data.py

/content/silent_speech
h5 copied
KenLM copied
tar restored
2026-09-09 11:27:08.955878: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-09 11:27:08.973215: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788953228.994421    2077 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788953229.001396    2077 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-09 11:27:09.022106: I tensorflow/core/platform/cpu_feature_guard.cc:210] Thi

In [ ]:
import glob, os, re, json, torch
root = "/content/drive/MyDrive/silent_speech"
hyb  = f"{root}/output_finetune_hybrid"

cands = sorted(glob.glob(f"{hyb}/model_*_last.pt"), key=os.path.getmtime)
if cands:
    src = cands[-1]
    state = torch.load(src, map_location="cpu", weights_only=False)
    print("resuming hybrid fine-tune from:", os.path.basename(src))
else:
    print("no hybrid checkpoint found -> building hybrid warm-start (TinyMyo transformer + your trained front-end)")
    os.chdir("/content/silent_speech")
    bbp = f"{root}/TinyMyo_backbone.pt"
    if not os.path.exists(bbp):
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        from architecture import EMGTransformer
        from data_utils import TextTransform
        sd = load_file(hf_hub_download("MatteoFasulo/TinyMyo", "pretraining/TinyMyo/TinyMyo.safetensors"))
        full = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in sd.items()}
        m = EMGTransformer(num_features=8, num_outs=len(TextTransform().chars)+1, in_chans=8,
                           embed_dim=192, n_layer=8, n_head=3, mlp_ratio=4)
        for k, v in m.state_dict().items():
            if "relative_positional" in k and k not in full:
                full[k] = torch.zeros_like(v)
        torch.save({"state_dict": full}, bbp)
    bb = torch.load(bbp, map_location="cpu", weights_only=False)["state_dict"]
    state = {k: v for k, v in bb.items() if k.startswith("blocks.")}
    scratch = sorted(glob.glob(f"{root}/output/model_*_best.pt"), key=os.path.getmtime)[-1]
    for k, v in torch.load(scratch, map_location="cpu", weights_only=False).items():
        if k.startswith(("conv_blocks.", "w_raw_in.", "w_out.")):
            state[k] = v
    print("front-end from:", os.path.basename(scratch))

state = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
idx = [int(re.match(r"blocks\.(\d+)\.", k).group(1)) for k in state if re.match(r"blocks\.(\d+)\.", k)]
n_layers = (max(idx) + 1) if idx else 8
torch.save({"state_dict": state}, f"{root}/resume_hybrid.pt")

p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg.update({"num_layers": n_layers, "start_training_from": f"{root}/resume_hybrid.pt", "freeze_blocks": False,
            "ckpt_directory": hyb, "num_epochs": 80, "eval_interval": 2,
            "num_workers": os.cpu_count(), "learning_rate": 2e-4})
json.dump(cfg, open(p, "w"), indent=4)
print(f"num_layers: {n_layers} | lr: 2e-4 | checkpoints save to: {hyb}")

resuming hybrid fine-tune from: model_20260909_101456_last.pt
num_layers: 8 | lr: 2e-4 | checkpoints save to: /content/drive/MyDrive/silent_speech/output_finetune_hybrid


In [ ]:
import os, json
def chk(l, path): print(f"{l:20}: {'OK' if os.path.exists(path) else 'MISSING <- fix this'}")
chk("h5",          "/content/data/Gaddy/h5/emg_dataset.h5")
chk("raw voiced",  "/content/data/Gaddy/emg_data/voiced_parallel_data")
chk("KenLM lm",    "/content/silent_speech/KenLM/lm.bin")
chk("lexicon",     "/content/silent_speech/KenLM/gaddy_lexicon.txt")
chk("resume ckpt", "/content/drive/MyDrive/silent_speech/resume_hybrid.pt")
cfg = json.load(open("/content/silent_speech/config/recognition_model.json"))
ok = cfg["ckpt_directory"].startswith("/content/drive/")
print(f"{'saves best to Drive':20}: {'OK -> ' + cfg['ckpt_directory'] if ok else 'NO <- fix ckpt_directory'}")

h5                  : OK
raw voiced          : OK
KenLM lm            : OK
lexicon             : OK
resume ckpt         : OK
saves best to Drive : OK -> /content/drive/MyDrive/silent_speech/output_finetune_hybrid


In [ ]:
%cd /content/silent_speech
!python recognition_model.py

/content/silent_speech
2026-09-09 11:29:26.929324: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-09 11:29:26.948707: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788953366.970611    2700 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788953366.976970    2700 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-09 11:29:26.998041: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to 

In [ ]:
import os, glob
os.chdir("/content/silent_speech")
os.environ["BEST"] = sorted(glob.glob("/content/drive/MyDrive/silent_speech/output_finetune_hybrid/model_*_best.pt"), key=os.path.getmtime)[-1]
print("evaluating:", os.environ["BEST"])
!python recognition_model.py --evaluate_saved "$BEST"

evaluating: /content/drive/MyDrive/silent_speech/output_finetune_hybrid/model_20260909_101456_best.pt
2026-09-09 11:30:17.995750: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-09 11:30:18.013714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788953418.036704    3117 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788953418.043000    3117 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-09 11:30:18.064115: I tensorflow/